# Unit 2 Hands-On ②: Taxi-v3 Q-Learning 실습

이 노트북은 **Hugging Face 딥 강화학습 강좌 Unit 2**의 두 번째 실습입니다.  
**Q-Learning** 알고리즘으로 `Taxi-v3` 환경에서 에이전트를 훈련하고,  
훈련 과정을 영상으로 기록하여 Google Drive에 저장합니다.

> FrozenLake(Unit2-①)와 동일한 Q-Learning 알고리즘을 사용하지만,  
> 상태 공간이 **500개**로 훨씬 크고, 보상 구조도 더 복잡합니다.

---
## 목차
1. 환경 설치
2. Google Drive 마운트
3. 가상 디스플레이 설정
4. 라이브러리 임포트
5. 환경 탐색
6. Q-테이블 초기화
7. 정책 함수 정의
8. 하이퍼파라미터 설정
9. 훈련 함수 정의 및 실행
10. 평가
11. 훈련 과정 영상 확인
12. Hugging Face Hub 업로드

---
## 1. 환경 설치

Taxi-v3 실행에 필요한 패키지를 설치합니다.  
`pickle5`는 Python 3.9+ 에 내장되어 있으므로 제외합니다.

In [1]:
# v3 호환 패키지 설치 (Unit1 커뮤니티 해결책 동일 적용)
# gymnasium[toy_text] 방식 대신 개별 설치로 충돌 방지
!pip install stable-baselines3==2.0.0a5 -q
!pip install gymnasium -q
!pip install pygame numpy huggingface_hub imageio imageio-ffmpeg tqdm -q

In [2]:
!sudo apt-get update -qq
!sudo apt-get install -y python3-opengl ffmpeg xvfb -qq
!pip install pyvirtualdisplay -q

---
## 2. Google Drive 마운트

Colab VM은 세션 종료 시 파일이 모두 삭제됩니다.  
Google Drive에 마운트하여 훈련 영상과 모델을 영구 보존합니다.

```
Google Drive/RL_Course/Unit2_Taxi/
├── training_videos/   ← 훈련 중간 단계별 영상
└── q-taxi.pkl         ← 최종 Q-테이블 모델
```

> 실행 시 Google 계정 인증 팝업이 뜹니다. 허용해주세요.

In [3]:
from google.colab import drive
import os

drive.mount('/content/drive')

# ✏️ 저장 폴더명을 원하는 대로 변경하세요.
DRIVE_BASE = "/content/drive/MyDrive/RL_Course/Unit2_Taxi/Certification"
VIDEO_DIR  = f"{DRIVE_BASE}/training_videos"
MODEL_DIR  = DRIVE_BASE

os.makedirs(VIDEO_DIR, exist_ok=True)
os.makedirs(MODEL_DIR, exist_ok=True)

print('✅ Drive 마운트 완료!')
print(f'   영상 저장 경로 : {VIDEO_DIR}')
print(f'   모델 저장 경로 : {MODEL_DIR}')

---
## 3. 가상 디스플레이 설정

Colab에는 물리적인 화면이 없으므로, rgb_array 렌더링을 위해 가상 디스플레이를 생성합니다.

In [4]:
from pyvirtualdisplay import Display

virtual_display = Display(visible=0, size=(1400, 900))
virtual_display.start()
print('✅ 가상 디스플레이 시작')

---
## 4. 라이브러리 임포트

| 라이브러리 | 역할 |
|---|---|
| `numpy` | Q-테이블 생성 및 수학 연산 |
| `gymnasium` | 강화학습 환경 |
| `random` | ε-greedy 탐색의 랜덤 행동 선택 |
| `imageio` | 프레임을 mp4로 저장 |
| `tqdm` | 훈련 진행률 표시 |
| `pickle` | Q-테이블 직렬화 저장 |

In [5]:
import numpy as np
import gymnasium as gym
import random
import imageio
import pickle
import glob

from tqdm.notebook import tqdm
from IPython.display import Video, display

---
## 5. 환경 탐색

### Taxi-v3 란?

5×5 격자 도시에서 택시가 승객을 픽업하여 목적지까지 데려다주는 환경입니다.

```
+---------+
|R: | : :G|    R, G, Y, B: 승객/목적지 후보 위치 4곳
| : | : : |    택시(노란색): 현재 택시 위치
| : : : : |    승객(파란색): 현재 승객 위치
| | : | : |    목적지(보라색): 목적지 위치
|Y| : |B: |
+---------+
```

**상태 공간**: 500개 (택시 위치 25 × 승객 위치 5 × 목적지 4)  
**행동 공간**: 6개

| 행동 | 의미 |
|:---:|---|
| 0 | 아래 이동 |
| 1 | 위 이동 |
| 2 | 오른쪽 이동 |
| 3 | 왼쪽 이동 |
| 4 | 승객 픽업 |
| 5 | 승객 하차 |

**보상 구조**:
- 목적지 하차 성공: +20
- 잘못된 픽업/하차: -10
- 이동 1스텝: -1 (빠를수록 유리)

In [8]:
# Taxi-v3 환경 생성 (rgb_array 모드)
env = gym.make('Taxi-v3', render_mode='rgb_array')

print('===== 관측 공간(Observation Space) =====')
print('크기:', env.observation_space.n, '→ 500가지 상태')

print('\n===== 행동 공간(Action Space) =====')
print('크기:', env.action_space.n, '→ 6가지 행동')

state_space  = env.observation_space.n   # 500
action_space = env.action_space.n        # 6
print(f'\n상태 공간: {state_space}, 행동 공간: {action_space}')

---
## 6. Q-테이블 초기화

Taxi-v3의 Q-테이블은 **500 × 6** 크기입니다.  
FrozenLake(16×4)보다 훨씬 크기 때문에 더 많은 에피소드가 필요합니다.

```
Q-테이블 (500×6)
       아래  위  오른  왼  픽업  하차
상태0  [ 0,  0,   0,   0,   0,   0 ]
상태1  [ 0,  0,   0,   0,   0,   0 ]
  ...
상태499[ 0,  0,   0,   0,   0,   0 ]
```

In [9]:
def initialize_q_table(state_space, action_space):
    """모든 Q값을 0으로 초기화한 Q-테이블 생성"""
    return np.zeros((state_space, action_space))

Qtable_taxi = initialize_q_table(state_space, action_space)
print(f'Q-테이블 크기: {Qtable_taxi.shape}  (상태 {state_space} × 행동 {action_space})')

---
## 7. 정책 함수 정의

FrozenLake와 동일한 Greedy / ε-Greedy 정책을 사용합니다.

```
랜덤값 > ε  →  활용: Q-테이블에서 최선의 행동 선택
랜덤값 ≤ ε  →  탐색: 완전 랜덤 행동 선택
```

Taxi-v3는 보상이 더 복잡하므로 `decay_rate=0.005`로 FrozenLake(0.0005)보다  
10배 빠르게 ε을 줄여 더 일찍 활용 위주로 전환합니다.

In [10]:
def greedy_policy(Qtable, state):
    """Q값이 최대인 행동 선택 (평가용)"""
    return np.argmax(Qtable[state][:])


def epsilon_greedy_policy(Qtable, state, epsilon):
    """ε 확률로 랜덤 탐색, 1-ε 확률로 탐욕적 선택 (훈련용)"""
    if random.uniform(0, 1) > epsilon:
        action = greedy_policy(Qtable, state)   # 활용
    else:
        action = env.action_space.sample()       # 탐색
    return action

---
## 8. 하이퍼파라미터 설정

| 파라미터 | 값 | FrozenLake와 비교 |
|---|---|---|
| `n_training_episodes` | 25,000 | FrozenLake는 10,000 |
| `learning_rate` | 0.7 | 동일 |
| `max_steps` | 99 | 동일 |
| `gamma` | 0.95 | 동일 |
| `decay_rate` | **0.005** | FrozenLake의 10배 (더 빠른 수렴) |
| `video_freq` | 5,000 | 5천 에피소드마다 영상 저장 |

> `eval_seed`는 강좌 공식 지정 시드로, **수정하지 마세요**.  
> 모든 수강생이 동일한 조건에서 평가받기 위한 값입니다.

In [11]:
# 훈련 파라미터
n_training_episodes = 25_000   # Taxi는 상태 공간이 커서 더 많은 에피소드 필요
learning_rate       = 0.7

# 평가 파라미터
n_eval_episodes = 100

# ⚠️ 아래 eval_seed는 수정하지 마세요 (강좌 공식 평가 시드)
eval_seed = [
    16, 54, 165, 177, 191, 191, 120, 80, 149, 178, 48, 38, 6,
    125, 174, 73, 50, 172, 100, 148, 146, 6, 25, 40, 68, 148,
    49, 167, 9, 97, 164, 176, 61, 7, 54, 55, 161, 131, 184, 51,
    170, 12, 120, 113, 95, 126, 51, 98, 36, 135, 54, 82, 45, 95,
    89, 59, 95, 124, 9, 113, 58, 85, 51, 134, 121, 169, 105, 21,
    30, 11, 50, 65, 12, 43, 82, 145, 152, 97, 106, 55, 31, 85,
    38, 112, 102, 168, 123, 97, 21, 83, 158, 26, 80, 63, 5, 81,
    32, 11, 28, 148,
]

# 환경 파라미터
env_id    = 'Taxi-v3'
max_steps = 99
gamma     = 0.95

# 탐색 파라미터
max_epsilon = 1.0
min_epsilon = 0.05
decay_rate  = 0.005    # FrozenLake(0.0005)보다 10배 빠른 감소

# 영상 저장 주기
video_freq = 5_000     # ✏️ 몇 에피소드마다 영상을 저장할지

---
## 9. 훈련 함수 정의 및 실행

### Q-Learning 업데이트 공식

$$Q(s,a) \leftarrow Q(s,a) + \alpha \left[ R + \gamma \max_{a'} Q(s',a') - Q(s,a) \right]$$

FrozenLake와 동일한 공식이지만, Taxi는 **보상이 음수(-1, -10)** 도 있어  
Q값이 음수 범위에서도 의미 있게 학습됩니다.

### 영상 저장 방식
`video_freq` 에피소드마다 현재 Q-테이블로 1 에피소드를 실행하여 mp4로 저장합니다.

In [12]:
def save_episode_video(env, Qtable, video_path, fps=2):
    """현재 Q-테이블로 1 에피소드를 실행하여 mp4로 저장"""
    frames = []
    state, _ = env.reset(seed=random.randint(0, 500))
    frames.append(env.render())
    terminated = truncated = False

    for _ in range(max_steps):
        action = greedy_policy(Qtable, state)
        state, _, terminated, truncated, _ = env.step(action)
        frames.append(env.render())
        if terminated or truncated:
            break

    imageio.mimsave(video_path, [np.array(f) for f in frames], fps=fps)


def train(n_training_episodes, min_epsilon, max_epsilon, decay_rate,
          env, max_steps, Qtable, video_freq, video_dir):
    """Q-Learning 훈련 루프. video_freq 에피소드마다 영상 저장."""

    for episode in tqdm(range(n_training_episodes), desc='훈련 진행'):

        # ε 지수 감소
        epsilon = min_epsilon + (max_epsilon - min_epsilon) * np.exp(-decay_rate * episode)

        state, _ = env.reset()
        terminated = truncated = False

        for _ in range(max_steps):
            action = epsilon_greedy_policy(Qtable, state, epsilon)
            new_state, reward, terminated, truncated, _ = env.step(action)

            # Q-Learning 업데이트
            Qtable[state][action] += learning_rate * (
                reward + gamma * np.max(Qtable[new_state]) - Qtable[state][action]
            )

            if terminated or truncated:
                break
            state = new_state

        # video_freq 에피소드마다 영상 저장
        if (episode + 1) % video_freq == 0:
            video_path = f'{video_dir}/episode_{episode+1:06d}.mp4'
            save_episode_video(env, Qtable, video_path)
            print(f'   🎬 [{episode+1:,} 에피소드] 영상 저장 → {video_path}')

    return Qtable

In [13]:
Qtable_taxi = train(
    n_training_episodes, min_epsilon, max_epsilon, decay_rate,
    env, max_steps, Qtable_taxi,
    video_freq=video_freq,
    video_dir=VIDEO_DIR
)

print('\n✅ 훈련 완료!')
print('Q-테이블 일부 (처음 5행):')
print(Qtable_taxi[:5])

---
## 10. 평가

공식 `eval_seed`로 100번의 에피소드를 실행하여 평균 보상을 측정합니다.

- 보상 구조: 목적지 하차 성공 +20, 잘못된 픽업/하차 -10, 이동 -1
- **평균 보상 7 이상**이면 잘 학습된 것으로 간주합니다.

In [14]:
def evaluate_agent(env, max_steps, n_eval_episodes, Q, seed):
    """n_eval_episodes 동안 에이전트를 실행하여 평균 보상 반환"""
    episode_rewards = []

    for episode in tqdm(range(n_eval_episodes), desc='평가 진행'):
        state, _ = env.reset(seed=seed[episode]) if seed else env.reset()
        total_reward = 0.0
        terminated = truncated = False

        for _ in range(max_steps):
            action = greedy_policy(Q, state)
            state, reward, terminated, truncated, _ = env.step(action)
            total_reward += reward
            if terminated or truncated:
                break

        episode_rewards.append(total_reward)

    return np.mean(episode_rewards), np.std(episode_rewards)


mean_reward, std_reward = evaluate_agent(env, max_steps, n_eval_episodes, Qtable_taxi, eval_seed)
print(f'\n평균 보상: {mean_reward:.2f} ± {std_reward:.2f}')
print('(7.0 이상이면 성공적으로 학습된 것)')

---
## 11. 훈련 과정 영상 확인

Drive에 저장된 영상을 순서대로 재생하여 학습 과정을 확인합니다.

| 훈련 단계 | 예상 행동 |
|---|---|
| 초반 (5,000 에피소드) | 무작위 이동, 자주 실패 |
| 중반 (15,000 에피소드) | 방향은 맞으나 픽업/하차 실수 |
| 후반 (25,000 에피소드) | 빠르고 정확하게 승객 이동 |

In [15]:
videos = sorted(glob.glob(f'{VIDEO_DIR}/*.mp4'))
print(f'총 {len(videos)}개의 영상이 저장되었습니다:\n')
for v in videos:
    ep = v.split('episode_')[1].split('.')[0]
    print(f'  {int(ep):,} 에피소드 시점: {v}')

In [16]:
for video_path in videos:
    ep = video_path.split('episode_')[1].split('.')[0]
    print(f'\n📽️  {int(ep):,} 에피소드 시점')
    display(Video(video_path, embed=True, width=400))

---
## 12. Hugging Face Hub 업로드

훈련된 Q-테이블을 HF Hub에 공유합니다.

### 사전 준비
1. [Hugging Face 계정 생성](https://huggingface.co/join)
2. [쓰기(write) 권한 토큰 발급](https://huggingface.co/settings/tokens)

In [ ]:
from huggingface_hub import login

# ✏️ 본인의 HF 토큰으로 교체하세요
# ⚠️ 토큰은 절대 외부에 공개하지 마세요!
login(token='hf_xxxxxxxxxxxxxxxxxxxxxxxx')

In [18]:
from huggingface_hub import HfApi, snapshot_download
from huggingface_hub.repocard import metadata_eval_result, metadata_save
from pathlib import Path
import datetime, json


def record_video(env, Qtable, out_directory, fps=2):
    """HF Hub용 최종 영상 1개 생성"""
    frames = []
    state, _ = env.reset(seed=random.randint(0, 500))
    frames.append(env.render())
    terminated = truncated = False
    for _ in range(max_steps):
        action = np.argmax(Qtable[state][:])
        state, _, terminated, truncated, _ = env.step(action)
        frames.append(env.render())
        if terminated or truncated:
            break
    imageio.mimsave(out_directory, [np.array(f) for f in frames], fps=fps)


def push_to_hub(repo_id, model, env, video_fps=2):
    """평가 → 영상 생성 → HF Hub 업로드 전체 파이프라인"""
    _, repo_name = repo_id.split('/')
    api = HfApi()

    repo_url = api.create_repo(repo_id=repo_id, exist_ok=True)
    repo_local_path = Path(snapshot_download(repo_id=repo_id))

    # Q-테이블 저장
    with open(repo_local_path / 'q-learning.pkl', 'wb') as f:
        pickle.dump(model, f)

    # 평가
    mean_reward, std_reward = evaluate_agent(
        env, model['max_steps'], model['n_eval_episodes'],
        model['qtable'], model['eval_seed']
    )

    # 결과 JSON 저장
    with open(repo_local_path / 'results.json', 'w') as f:
        json.dump({'env_id': model['env_id'], 'mean_reward': mean_reward,
                   'n_eval_episodes': model['n_eval_episodes'],
                   'eval_datetime': datetime.datetime.now().isoformat()}, f)

    # 모델 카드 메타데이터
    env_name = model['env_id']
    metadata = {'tags': [env_name, 'q-learning', 'reinforcement-learning']}
    eval_meta = metadata_eval_result(
        model_pretty_name=repo_name, task_pretty_name='reinforcement-learning',
        task_id='reinforcement-learning', metrics_pretty_name='mean_reward',
        metrics_id='mean_reward', metrics_value=f'{mean_reward:.2f} +/- {std_reward:.2f}',
        dataset_pretty_name=env_name, dataset_id=env_name,
    )
    metadata = {**metadata, **eval_meta}

    readme_path = repo_local_path / 'README.md'
    readme = readme_path.read_text(encoding='utf8') if readme_path.exists() else \
        f'# Q-Learning Agent playing {env_name}\n'
    readme_path.write_text(readme, encoding='utf-8')
    metadata_save(readme_path, metadata)

    # 영상 생성 및 업로드
    record_video(env, model['qtable'], repo_local_path / 'replay.mp4', video_fps)
    api.upload_folder(repo_id=repo_id, folder_path=repo_local_path, path_in_repo='.')
    print('\n✅ 업로드 완료:', repo_url)

In [19]:
model = {
    'env_id':              env_id,
    'max_steps':           max_steps,
    'n_training_episodes': n_training_episodes,
    'n_eval_episodes':     n_eval_episodes,
    'eval_seed':           eval_seed,
    'learning_rate':       learning_rate,
    'gamma':               gamma,
    'max_epsilon':         max_epsilon,
    'min_epsilon':         min_epsilon,
    'decay_rate':          decay_rate,
    'qtable':              Qtable_taxi,
}

In [20]:
# ✏️ 본인의 HF 사용자명과 저장소 이름을 입력하세요.
username  = 'DitDahDitDit'              # ← HF 사용자명 입력
repo_name = 'q-Taxi-v3'

push_to_hub(
    repo_id=f'{username}/{repo_name}',
    model=model,
    env=env
)